<a href="https://colab.research.google.com/github/YANHONGLU/Value-Index-Construction-Rebalancing/blob/main/Value_Index_Construction_%26_Rebalancing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Value Index Construction and Rebalancing**

This project build and test a reusable Python class to construct & rebalance a value index of 50 stocks, including 4 steps:

1.Imports

2.IndexRebalance Class

3.Run 3 Rebalances

4.Test

## **1. Imports**

In [1]:
# AI assistance disclosure: ChatGPT and Claude assisted with code development.
import numpy as np
import pandas as pd
from scipy.optimize import minimize

## **2. IndexRebalance Class**

`__init__()` initialises the index

`load_data()` reads & validates stock data for a specific date

`rebalance()` selects 50 stocks, optimises their weights & exports the results

**Note:**

Keep previous stocks for the buffer rule and save past results.

Clear current data to avoid using old data in the next rebalance.

In [2]:
class IndexRebalance:
    def __init__(self):
        self.data = None       # Current rebalance data
        self.previous = set()  # Previous rebalance(stock codes)
        self.results = []      # All rebalance results

    def load_data(self, filename, date):
        self.data = None  # Clear any previously loaded data
        data = pd.read_excel(filename,sheet_name="Start Universe")

        # 1) Date:Convert
        date = pd.Timestamp(date)
        data["Ref Date"] = pd.to_datetime(data["Ref Date"], errors="raise")  #❗️errors="coerce" silently converts invalid dates to NaT

        # Filter
        data = data.loc[data["Ref Date"] == date].copy()

        # 2) Stock: Check numbers
        if len(data) < 50 or data["RIC"].duplicated().any():
                  raise ValueError("Too few stocks or duplicate RIC")

        # 3) Numeric Columns:Convert
        for col in ["Sector Code", "FCap Wt", "Z_Value"]:
                  data[col] = pd.to_numeric(data[col], errors="raise")
        # Numeric Columns:Check
        numeric = data[["Sector Code", "FCap Wt", "Z_Value"]]
        if data.isna().any().any() or not np.isfinite(numeric).all().all() or (data["FCap Wt"] <= 0).any():
                  raise ValueError("Invalid stock data")

        # 4) Save
        self.data = data

    def rebalance(self, filename):
        if self.data is None:
            raise ValueError("Load data first")
        # 1) Index Construction Rules
        # 1.Select top 40 stocks
        ranked = self.data.sort_values(["Z_Value", "RIC"], ascending=[False, True])
        selected = ranked.head(40)

        # 2.Add old stocks from buffer (ranks 41–60)
        buffer = ranked.iloc[40:60]
        old = buffer[buffer["RIC"].isin(self.previous)]
        selected = pd.concat([selected, old.head(10)])

        # 3. Fill remaining places with unselected stocks to reach 50
        new = ranked[ ~ranked["RIC"].isin(selected["RIC"])]
        selected = pd.concat([selected, new.head(50 - len(selected))])

        # 2) Constituent Weighting Scheme
        # Calculate & normalise uncapped weights and Check
        raw_weight = selected["FCap Wt"] * (1 + selected["Z_Value"])
        if (raw_weight <= 0).any():
            raise ValueError("Invalid initial weights")
        initial = raw_weight / raw_weight.sum()

        # 1.Max stock weight: Min(5%, 20 * FCap Wt)
        upper = np.minimum(0.05, 20 * selected["FCap Wt"])

        # 2. Min stock weight: 0.05%
        lower = np.full(50, 0.0005)  # Minimum individual weight: 0.05%

        # 3.Total weight: 100%
        constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1}]

        # 4. Max sector weight: 50%
        sectors = [s == selected["Sector Code"]  for s in set(selected["Sector Code"])]
        for sector in sectors:
            constraints.append({"type": "ineq", "fun": lambda w, s=sector: 0.5 - s @ w})


        # 3) Objective
        def objective(w):
            return np.sum((w - initial) ** 2 / initial)

        # Optimiser:finding optimal weights
        result = minimize(objective,
                          initial.values,# np.full(50, 0.02) same result, because the problem is convex; initial is more natural
                          bounds=zip(lower, upper),
                          method="SLSQP",
                          constraints=constraints,
                          options={"ftol": 1e-12, "maxiter": 3000})

        # Check
        if not result.success:
              raise ValueError("Optimisation failed")

        # Save
        w = result.x
        selected["Weight"] = w

        # Export
        selected = selected[["Ref Date", "Company Name", "RIC", "Sector Code", "FCap Wt", "Z_Value", "Weight"]]
        pd.concat(self.results + [selected]).to_excel(filename, sheet_name="Output Sheet", index=False)

        # Update Index State
        self.results.append(selected)
        self.previous = set(selected["RIC"])
        self.data = None  # Require fresh data for the next rebalance

## **3. Run 3 Rebalances**

In [3]:
# Initialise once to keep the buffer history
index = IndexRebalance()
assert index.data is None and not index.previous and not index.results

In [4]:
for date in ["2017-06-30", "2017-12-31", "2018-06-30"]:
    index.load_data("Python Exercise v4.xlsx", date)
    index.rebalance("Index_Output.xlsx")

## **4. Test**

#### General Checks

In [5]:
# Test 1: Check 50 stocks and unique RICs in each rebalance
for result in index.results:
    assert len(result) == 50 and result["RIC"].is_unique

In [6]:
# Test 2: Check error handling for a missing date
try:
    index.load_data("Python Exercise v4.xlsx", "2020-01-01")
except ValueError:
    assert index.data is None
else:
    raise AssertionError("Missing date should fail")


#### Index Rules Checks

In [7]:
# Test 3: Check stock and sector weight limits
for result in index.results:
    assert (result["Weight"] <= np.minimum(0.05, 20 * result["FCap Wt"]) + 1e-7).all() # Max stock weight: Min(5%, 20 * FCap Wt)
    assert (result["Weight"] >= 0.0005 - 1e-7).all()  # Min stock weight: 0.05%

    assert abs(result["Weight"].sum() - 1) < 1e-7 # Total weight: approximately 100%

    assert (result.groupby("Sector Code")["Weight"].sum() <= 0.5 + 1e-7).all()  # Max sector weight: 50%

In [8]:
#Test 4: Buffer Rule
# 1) Prepare Synthetic Data
# 1.Create 80 sample stock codes: S01 to S80
codes = [f"S{i:02d}" for i in range(1, 81)]

# 2.Create sample stock data
sample = pd.DataFrame({
    "Ref Date": pd.Timestamp("2020-06-30"),
    "Company Name": codes,
    "RIC": codes,
    "Sector Code": [100, 200, 300, 400, 500] * 16,
    "FCap Wt": [1 / 80] * 80,
    "Z_Value": np.linspace(3, -0.5, 80)
})

# 3.Create temporary folders
from tempfile import TemporaryDirectory

In [9]:
#2) Run Buffer Test
with TemporaryDirectory() as folder:
    sample.to_excel(f"{folder}/input.xlsx", sheet_name="Start Universe", index=False)
    # 1.Prepare test data
    test = IndexRebalance()

    # 2.Run test rebalance
    test.previous = {"S60"}
    test.load_data(f"{folder}/input.xlsx", "2020-06-30")
    test.rebalance(f"{folder}/output.xlsx")

    # 3.Check
    assert set(test.results[-1]["RIC"]) == set(codes[:49]) | {"S60"}  # Expected constituents: S01–S49 + S60


### Output Check


In [10]:
#Test 5: Output
assert len(pd.read_excel("Index_Output.xlsx", sheet_name="Output Sheet")) == 150  # 3 rebalance dates × 50 stocks = 150 rows

## **Reflection**

#### 1.Data Quality

- Overall good quality. No missing values or duplicate RICs.

- Some company names have formatting issues: 17 rows (8 stocks) contain "?"  e.g. "Shenzhen Zhenye?Group? Co.,Ltd." → use `RIC` as the stock ID, so results are not affected.

#### 2.Demo Comparison

- Same 50 stocks, with small weight differences (max about 0.004%), likely due to different solver settings.

#### 3.Improvements

- Make fixed numbers (50, top 40, top 60, 5%) into parameters

- Add more test e.g. edge-case tests(conflicting constraints),stability tests

